In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import kagglehub


path = kagglehub.dataset_download("tushar5harma/plant-village-dataset-updated")

print("Path to dataset files:", path)

Mounting files to /kaggle/input/datasets/tushar5harma/plant-village-dataset-updated...
Path to dataset files: /kaggle/input/datasets/tushar5harma/plant-village-dataset-updated


In [3]:
import os
import shutil

# Pointing to the Kaggle input path
original_dataset_path = path
new_dataset_path = "/kaggle/working/organized_dataset"

print("Starting reorganization. Ignoring hidden system folders...")

for plant_name in os.listdir(original_dataset_path):
    plant_path = os.path.join(original_dataset_path, plant_name)
    if not os.path.isdir(plant_path): continue

    for split in os.listdir(plant_path):
        split_path = os.path.join(plant_path, split)
        if not os.path.isdir(split_path): continue

        for disease_name in os.listdir(split_path):
            disease_path = os.path.join(split_path, disease_name)
            if not os.path.isdir(disease_path): continue

            class_name = f"{plant_name}___{disease_name}"
            target_dir = os.path.join(new_dataset_path, split, class_name)
            os.makedirs(target_dir, exist_ok=True)

            # Copy images
            for img_file in os.listdir(disease_path):
                src = os.path.join(disease_path, img_file)
                
                # THE FIX: Check if it is actually a file. 
                # This ignores hidden folders like .ipynb_checkpoints
                if not os.path.isfile(src): 
                    continue
                
                dst = os.path.join(target_dir, img_file)
                shutil.copy(src, dst)

print(f"Reorganization complete! Clean data is ready at: {new_dataset_path}")

Starting reorganization. Ignoring hidden system folders...
Reorganization complete! Clean data is ready at: /kaggle/working/organized_dataset


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# --- 1. SETUP & AUGMENTATION ---
# THE FIX: Point to the Kaggle working directory instead of Colab's content directory
TRAIN_DIR = "/kaggle/working/organized_dataset/Train"
BATCH_SIZE = 32
EPOCHS = 20

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
num_classes = len(train_dataset.classes)
print(f"Training on {num_classes} unique classes with gentle Data Augmentation.")


# --- 2. THE UPGRADED DEEP CNN ARCHITECTURE ---
class PlantDiseaseCNN_Advanced(nn.Module):
    def __init__(self, num_classes):
        super(PlantDiseaseCNN_Advanced, self).__init__()

        self.features = nn.Sequential(
            # Block 1: 32 Channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: 64 Channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: 128 Channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 4: 256 Channels
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),

            # Global Average Pooling (Replaces the massive Flattening step)
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = PlantDiseaseCNN_Advanced(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# NEW: Automatically reduces learning rate by 50% every 5 epochs
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# --- 3. THE TRAINING LOOP ---
print("Starting Advanced Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Step the scheduler at the end of each epoch
    scheduler.step()

    current_lr = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {running_loss/len(train_loader):.4f} | LR: {current_lr}")

torch.save(model.state_dict(), "suspect_model.pth")
print("Advanced Model saved as 'suspect_model.pth'. Phase 1 Complete.")

Training on: cuda
Training on 29 unique classes with gentle Data Augmentation.
Starting Advanced Training...
Epoch 1/30 - Loss: 1.4844 | LR: 0.001
Epoch 2/30 - Loss: 0.6188 | LR: 0.001
Epoch 3/30 - Loss: 0.4215 | LR: 0.001
Epoch 4/30 - Loss: 0.3295 | LR: 0.001
Epoch 5/30 - Loss: 0.2612 | LR: 0.0005
Epoch 6/30 - Loss: 0.1725 | LR: 0.0005
Epoch 7/30 - Loss: 0.1498 | LR: 0.0005
Epoch 8/30 - Loss: 0.1367 | LR: 0.0005
Epoch 9/30 - Loss: 0.1269 | LR: 0.0005
Epoch 10/30 - Loss: 0.1188 | LR: 0.00025
Epoch 11/30 - Loss: 0.0863 | LR: 0.00025
Epoch 12/30 - Loss: 0.0788 | LR: 0.00025
Epoch 13/30 - Loss: 0.0772 | LR: 0.00025
Epoch 14/30 - Loss: 0.0700 | LR: 0.00025
Epoch 15/30 - Loss: 0.0675 | LR: 0.000125
Epoch 16/30 - Loss: 0.0559 | LR: 0.000125
Epoch 17/30 - Loss: 0.0513 | LR: 0.000125
Epoch 18/30 - Loss: 0.0519 | LR: 0.000125
Epoch 19/30 - Loss: 0.0491 | LR: 0.000125
Epoch 20/30 - Loss: 0.0501 | LR: 6.25e-05
Epoch 21/30 - Loss: 0.0404 | LR: 6.25e-05
Epoch 22/30 - Loss: 0.0395 | LR: 6.25e-05
Epo

In [6]:
import torch
from torchvision import datasets, transforms
import torchvision.transforms.functional as F
from torchvision.utils import save_image
import random
import os
import unittest

print("Initializing Interrogation Room...")

# THE FIX: Point to the Kaggle working directory
defect_folder = "/kaggle/working/defect_reports"
os.makedirs(defect_folder, exist_ok=True)
print(f"Defect tracking folder ready at: {defect_folder}")

# THE FIX: We must use a CLEAN transform for testing. No random augmentations!
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# THE FIX: Point to the Kaggle working directory
VAL_DIR = "/kaggle/working/organized_dataset/Val"
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=test_transform)
class_names = val_dataset.classes

# THE FIX: Load the NEW advanced model architecture
test_model = PlantDiseaseCNN_Advanced(num_classes).to(device)
test_model.load_state_dict(torch.load("suspect_model.pth"))
test_model.eval()

# =====================================================================
# PART 2: THE MUTATIONS (Module 4: Boundary Value Analysis)
# =====================================================================
# Standard Equivalence Class Tests
def mutate_flip(img):
    return F.hflip(img)

def mutate_darken(img):
    return F.adjust_brightness(img, 0.5) # 50% darker

def mutate_rotate(img):
    return F.rotate(img, 25) # Normal 25-degree tilt

# Boundary Value Tests (Extreme edge cases)
def mutate_boundary_pitch_black(img):
    return F.adjust_brightness(img, 0.01) # 99% darker (Almost invisible)

def mutate_boundary_blinding_light(img):
    return F.adjust_brightness(img, 2.5) # 150% brighter (Washed out)

def mutate_boundary_full_rotation(img):
    return F.rotate(img, 359) # 359 degrees (Should be visually identical to 0)


# =====================================================================
# PART 3: UNIT TESTING (Module 4: Levels of Testing - Unit Test)
# =====================================================================
class TestMutationPipeline(unittest.TestCase):
    def setUp(self):
        # Create a fake 128x128 image tensor to test our logic
        self.dummy_image = torch.rand(3, 128, 128)

    def test_flip_integrity(self):
        # Proves flipping doesn't corrupt the data shape
        result = mutate_flip(self.dummy_image)
        self.assertEqual(result.shape, self.dummy_image.shape)

    def test_darken_logic(self):
        # Proves the darken function mathematically lowers pixel values
        result = mutate_darken(self.dummy_image)
        self.assertTrue(torch.mean(result) < torch.mean(self.dummy_image))

print("\n--- Running White-Box Unit Tests on Testing Tools ---")
# 'exit=False' prevents Colab/Kaggle from stopping the entire script after the unit test
unittest.main(argv=[''], exit=False)


# =====================================================================
# PART 4: SYSTEM PIPELINE (Module 4: System & Black Box Testing)
# =====================================================================
print("\n--- Starting Black-Box AI Stress Test ---")
total_tests = 200 # Number of images to test
test_results = {
    "Flip": 0, "Darken": 0, "Rotate": 0,
    "Pitch_Black (Boundary)": 0, "Blinding_Light (Boundary)": 0, "Rotate_359 (Boundary)": 0
}

for _ in range(total_tests):
    # 1. Grab a random image
    idx = random.randint(0, len(val_dataset) - 1)
    img, true_label = val_dataset[idx]

    # 2. Establish the baseline prediction
    with torch.no_grad():
        original_output = test_model(img.unsqueeze(0).to(device))
        original_pred = torch.argmax(original_output, dim=1).item()

    # 3. Apply all mutations
    mutations = {
        "Flip": mutate_flip(img),
        "Darken": mutate_darken(img),
        "Rotate": mutate_rotate(img),
        "Pitch_Black (Boundary)": mutate_boundary_pitch_black(img),
        "Blinding_Light (Boundary)": mutate_boundary_blinding_light(img),
        "Rotate_359 (Boundary)": mutate_boundary_full_rotation(img)
    }

    # 4. Interrogate the AI on the mutated images
    for mutation_name, mutated_img in mutations.items():
        with torch.no_grad():
            mutated_output = test_model(mutated_img.unsqueeze(0).to(device))
            mutated_pred = torch.argmax(mutated_output, dim=1).item()

        # 5. Log the failure if the AI changes its mind
        if original_pred != mutated_pred:
            test_results[mutation_name] += 1

            # Save visual evidence of the bug
            actual_disease_name = class_names[true_label]
            # Added a random number to the filename so bugs don't overwrite each other
            bug_filename = f"{defect_folder}/FAIL_{mutation_name}_{actual_disease_name}_{random.randint(1000,9999)}.png"
            save_image(mutated_img, bug_filename)


# =====================================================================
# PART 5: STATISTICAL QA (Module 4: Statistical Quality Assurance)
# =====================================================================
print("\n" + "=" * 50)
print("MODULE 4 FINAL SOFTWARE METRICS REPORT")
print("=" * 50)
print(f"Total Base Images Interrogated: {total_tests}")
print(f"Total Individual Tests Executed: {total_tests * len(test_results)}\n")

for test_name, fail_count in test_results.items():
    fail_rate = (fail_count / total_tests) * 100
    print(f"Failed '{test_name}' Test: {fail_count} times | Failure Rate: {fail_rate:.1f}%")

print(f"\nDebug artifacts (failed images) have been saved to '{defect_folder}'.")
print("=" * 50)

./usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
.
----------------------------------------------------------------------
Ran 2 tests in 0.019s

OK


Initializing Interrogation Room...
Defect tracking folder ready at: /kaggle/working/defect_reports

--- Running White-Box Unit Tests on Testing Tools ---

--- Starting Black-Box AI Stress Test ---

MODULE 4 FINAL SOFTWARE METRICS REPORT
Total Base Images Interrogated: 200
Total Individual Tests Executed: 1200

Failed 'Flip' Test: 2 times | Failure Rate: 1.0%
Failed 'Darken' Test: 7 times | Failure Rate: 3.5%
Failed 'Rotate' Test: 2 times | Failure Rate: 1.0%
Failed 'Pitch_Black (Boundary)' Test: 187 times | Failure Rate: 93.5%
Failed 'Blinding_Light (Boundary)' Test: 110 times | Failure Rate: 55.0%
Failed 'Rotate_359 (Boundary)' Test: 0 times | Failure Rate: 0.0%

Debug artifacts (failed images) have been saved to '/kaggle/working/defect_reports'.


In [7]:
import shutil
from IPython.display import FileLink, display

print("Packaging defect reports into a zip file...")
# Compress the folder into a single zip file for easy downloading
shutil.make_archive('/kaggle/working/final_defect_reports', 'zip', '/kaggle/working/defect_reports')

print("\n✅ Packaging complete! Click the links below to download your files:")
print("-" * 50)

# Generate clickable download links right in the notebook
display(FileLink('final_defect_reports.zip'))
display(FileLink('suspect_model.pth'))

Packaging defect reports into a zip file...

✅ Packaging complete! Click the links below to download your files:
--------------------------------------------------


/kaggle/working/final_defect_reports.zip

/kaggle/working/suspect_model.pth